# Fact_PartAssociation Build

Part×Part market-basket association, computed weekly (off the daily
critical path). See `docs/superpowers/specs/2026-08-27-associated-parts-design.md`
and `docs/superpowers/plans/2026-08-27-associated-parts-recommended-parts.md`
for the full design and validation history.

Reads `InTrans_Incremental` via native Spark (this notebook's attached
default lakehouse), does the actual pairwise aggregation in DuckDB
(same engine already proven safe for this shape of self-join elsewhere
in this repo), and writes back via `.save("Tables/...")` (not
`saveAsTable()`) to preserve PascalCase in the stored Delta table name.

In [ ]:
%pip install duckdb

In [ ]:
import duckdb

BASKET_CAP = 25
MIN_COOCCURRENCE = 10

# Native Spark read of the attached lakehouse — no OneLake/az-login
# workaround needed inside a Fabric notebook.
intrans_df = spark.sql("""
    SELECT Franchise, Branch, RONumber, PartNumber
    FROM InTrans_Incremental
    WHERE Type = 'I' AND Qty > 0
      AND TransDatetime >= add_months(current_date(), -24)
      AND PartNumber IS NOT NULL AND PartNumber <> ''
""")
intrans_pdf = intrans_df.toPandas()
print(f"Filtered InTrans rows pulled into pandas: {len(intrans_pdf):,}")

In [ ]:
con = duckdb.connect()
con.register("intrans", intrans_pdf)

result_pdf = con.execute(f"""
    WITH baskets AS (
        SELECT Franchise, Branch, RONumber, PartNumber
        FROM intrans
        GROUP BY Franchise, Branch, RONumber, PartNumber
    ),
    basket_sizes AS (
        SELECT Franchise, Branch, RONumber, COUNT(*) AS DistinctParts
        FROM baskets GROUP BY Franchise, Branch, RONumber
    ),
    capped_baskets AS (
        SELECT b.*
        FROM baskets b
        INNER JOIN basket_sizes s
          ON b.Franchise = s.Franchise AND b.Branch = s.Branch AND b.RONumber = s.RONumber
        WHERE s.DistinctParts <= {BASKET_CAP}
    ),
    pairs AS (
        SELECT a.Franchise, a.PartNumber AS PartA, b.PartNumber AS PartB
        FROM capped_baskets a
        INNER JOIN capped_baskets b
          ON a.Franchise = b.Franchise AND a.Branch = b.Branch AND a.RONumber = b.RONumber
         AND a.PartNumber <> b.PartNumber
    ),
    co_occurrence AS (
        SELECT Franchise, PartA, PartB, COUNT(*) AS CoOccurrenceCount
        FROM pairs GROUP BY Franchise, PartA, PartB
    ),
    part_totals AS (
        SELECT Franchise, PartNumber, COUNT(*) AS InvoiceCount
        FROM capped_baskets GROUP BY Franchise, PartNumber
    ),
    total_invoices AS (
        SELECT Franchise, COUNT(*) AS TotalInvoiceCount
        FROM basket_sizes GROUP BY Franchise
    )
    SELECT
        c.Franchise, c.PartA, c.PartB, c.CoOccurrenceCount,
        ta.InvoiceCount AS AnchorInvoiceCount,
        tb.InvoiceCount AS AssociatedInvoiceCount,
        ti.TotalInvoiceCount
    FROM co_occurrence c
    INNER JOIN part_totals ta ON ta.Franchise = c.Franchise AND ta.PartNumber = c.PartA
    INNER JOIN part_totals tb ON tb.Franchise = c.Franchise AND tb.PartNumber = c.PartB
    INNER JOIN total_invoices ti ON ti.Franchise = c.Franchise
    WHERE c.CoOccurrenceCount >= {MIN_COOCCURRENCE}
""").df()

print(f"Fact_PartAssociation rows: {len(result_pdf):,}")

# Row-count sanity guard: this notebook runs weekly, unattended, and
# overwrites a real Lakehouse table. 1,000 is well below the validated
# ~44,326-row expectation (room for legitimate future data drift) but
# high enough to catch a broken read (bad filter, upstream pipeline
# failure, schema rename) before it clobbers the good table.
assert len(result_pdf) > 1000, f"Fact_PartAssociation row count ({len(result_pdf):,}) is suspiciously low — aborting write."

In [ ]:
result_df = spark.createDataFrame(result_pdf)

# .save("Tables/...") — NOT saveAsTable() — preserves PascalCase.
# saveAsTable() lowercases the stored Delta table name in the Hive
# metastore, which breaks the semantic model's exact-case SQL Analytics
# Endpoint lookup (documented gotcha, confirmed twice previously in this repo).
result_df.write.format("delta").mode("overwrite").save("Tables/Fact_PartAssociation")
print("Wrote Fact_PartAssociation.")